In [ ]:
"""
Prereqs: download 10 vids/directory in all_dirs via download_ff_data.sh
"""
from collections import defaultdict
import re
import os
import pandas as pd

def group_by_video(directory):
    groups = defaultdict(list)
    for video_name in os.listdir(directory):
        video_path = os.path.join(directory, video_name)
        if not os.path.isdir(video_path):
            continue
        frames = sorted(f for f in os.listdir(video_path) if f.endswith('.png'))
        if frames:
            # change: doesn't prepend video name to frame filename
            groups[video_name] = [f for f in frames]
    
    result = []
    for key, frames in groups.items():
        result.append([key, frames])
    return result

In [ ]:
all_dirs = ['FF_data/real/original_sequences/youtube/c40/images', 
            'FF_data/fake/manipulated_sequences/Deepfakes/c40/images',
            'FF_data/DFD_real/original_sequences/actors/c40/images',
            'FF_data/DFD_fake/manipulated_sequences/DeepFakeDetection/c40/images',
            'FF_data/Face2Face/manipulated_sequences/Face2Face/c40/images',
            'FF_data/FaceSwap/manipulated_sequences/FaceSwap/c40/images',
            'FF_data/NeuralTextures/manipulated_sequences/NeuralTextures/c40/images'
            ]

'''
0 - Authentic
1 - Deepfake
'''
# same thing as the original version, but combines the root dir and vidpath to
# make ID'ing unique vids easier
def create_datamap(all_dirs):
    data = {
        'video_path': []
    }
    for i in range(10):
        data['frame_' + str(i)] = []
    
    data['label'] = []

    for dir in all_dirs:
        label = 1
        if 'original' in dir:
            label = 0
        # list of videos in dir, along with a list of frame filenames with them 
        #[[vid, [vid/fr1.png, vid/fr2.png, ...]], ...]
        video_map = group_by_video(dir)
        for video, frames in video_map:
            i = 0
            for j in range(10, len(frames), 10):
                split = frames[i:j]
                i = j
                data['video_path'].append(os.path.join(dir, video))
                data['label'].append(label)
                for k in range(len(split)):
                    data['frame_' + str(k)].append(split[k])
    return data

data_map = create_datamap(all_dirs)
df = pd.DataFrame(data_map)
df.to_csv('dataset_splits2.csv', index=False) # for testing dataset


1


In [ ]:

# from sklearn.model_selection import train_test_split
df = pd.read_csv("dataset_splits2.csv")
vidnames = list(set(df["video_path"]))
print(f"Number of videos (should be 70): {len(vidnames)}") 
real_vids = [name for name in vidnames if "original" in name]
fake_vids = [name for name in vidnames if "original" not in name]

train_vids = real_vids[0:5] + fake_vids[0:5]
test_vids = [real_vids[6], fake_vids[6]]
# TODO use train/test split to make even proportional split of vids

train_frames = df.loc[[vid in train_vids for vid in df["video_path"]]]
test_frames = df.loc[[vid in test_vids for vid in df["video_path"]]]



In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from PIL import Image
from torchvision import transforms
import os

if not torch.cuda.is_available():
    print("!!!\n\tCUDA not available. Everything ok?\n!!!")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FrameDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform or transforms.ToTensor()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        """
        Returns a clip tensor of shape (3, T, H, W) = (3, 10, 500, 500).
        The DataLoader will stack these into (B, 3, T, H, W).
        """
        vidpath = self.df.iloc[idx, 0]
        tensors = []
        for i in range(10):
            fname = self.df.iloc[idx, i + 1]  # cols 1–10 are frame_0–frame_9
            path = os.path.join(vidpath, fname)
            img = Image.open(path).convert('RGB').resize((500, 500))
            tensors.append(self.transform(img)) # (3, 500, 500)
        # TODO normalize pixel vals
        clip = torch.stack(tensors, dim=1).to(dtype=torch.float32) # (3, T, 500, 500)
        label = torch.Tensor([self.df.iloc[idx, -1]])
        return clip, label


train_dataset = FrameDataset(train_frames)
train_loader = DataLoader(train_dataset, batch_size=4)
test_dataset = FrameDataset(test_frames)


cuda


In [ ]:
def test_model(model, data, device='cpu', do_print=False):
    model.to(device=device)
    model.eval()
    total = 0
    correct = 0
    labels = ["Real", "Fake"]
    record = []
    output = []
    for i, (frames, label) in enumerate(data):
        frames = frames.unsqueeze(0).to(device) # add batch dimension
        output = model(frames)[0] # since batch size is 1, we will get only one output
        y_pred = torch.round(output).item()
        y_actual = label
        record.append((y_pred, y_actual))
        if y_actual == y_pred:
            correct = correct+1
        total = total+1
    output.append(f"Accuracy = {correct / total * 100:.4f} %")
    for i in range(2):
        tps = sum([1 for x in record if x[0] == i and x[1] == i])
        fps = sum([1 for x in record if x[0] == i and x[1] != i])
        fns = sum([1 for x in record if x[0] != i and x[1] == i])
        tns = sum([1 for x in record if x[0] != i and x[1] != i])
        guesses = [x[0] for x in record if x[1] == i]
        # represents what the model guessed when this was the actual label
        counts = [guesses.count(j) for j in range(2)]
        if tps == 0:
            precision = 0.0
            recall = 0.0
            f_score = 0.0
        else:
            precision = tps / (tps + fps)
            recall = tps / (tps + fns)
            f_score = 2 * precision * recall / (precision + recall)
        output.append(f"  {labels[i]} ({i}):\n    F Score: {f_score:.4f}\n    Guesses: {counts}\n    Precision: {precision:.4f}\n    Recall: {recall:.4f}")
        if i == 1:
            output.append(f"\tPred Pos.\tPred Neg.\n\tReal Pos.\t{tps}\t{fns}\n\tReal Neg.\t{fps}\t{tns}")
    output = "\n".join(output)
    if do_print:
        print(output)
    return correct / total * 100, output

In [ ]:
import pickle
from tqdm import tqdm
def train_model(model, train_loader, criterion, optimizer, num_epochs, test_data, output_dir, device='cpu'):
    os.makedirs(os.path.join(output_dir, "checkpoints"), exist_ok=True)
    model.to(device)
    losses = []
    vals = []
    best_val = -1
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1} / {num_epochs}")
        for frames, labels in progress_bar:
            # Moving the data to GPU if available
            frames, labels = frames.to(device=device), labels.to(device=device)
            optimizer.zero_grad()
            outputs = model(frames)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            losses.append(loss.item())
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")
        val, output = test_model(model, test_data, device)
        vals.append(val)
        with open(os.path.join(output_dir, "stats.pkl"), "wb") as f:
            pickle.dump({"epoch": epoch + 1, "loss": losses, "val": vals}, f)
        
        if val > best_val:
            print(f" ! New best val: {val:.4f} !")
            best_val = val
            with open(os.path.join(output_dir, "checkpoints", f"epc_{epoch + 1:04d}_val_{val:.3f}"), "wb") as f:
                torch.save({"model": model, "optimizer": optimizer, "output": output}, f)
        elif epoch + 1 % 50 == 0:
            with open(os.path.join(output_dir, "checkpoints", f"epch_{epoch + 1:04d}_val_{val:.3f}"), "wb") as f:
                torch.save({"model": model, "optimizer": optimizer, "output": output}, f)
    return losses

In [ ]:
from classifier_model import AIClassifier
lr = .0005
epochs = 3

criterion = nn.BCELoss().to(device=device)
model = AIClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)


losses = train_model(model, train_loader, criterion, optimizer, epochs, test_dataset, "output-tenfiles", device=device)

Epoch 1 / 3: 100%|██████████| 164/164 [06:58<00:00,  2.55s/it]


Epoch 1/3, Loss: 6850.6389
 ! New best val: 49.0909 !
Accuracy = 49.0909 %
  Real (0):
    F Score: 0.6585
    Guesses: [54, 0]
    Precision: 0.4909
    Recall: 1.0000
  Fake (1):
    F Score: 0.0000
    Guesses: [56, 0]
    Precision: 0.0000
    Recall: 0.0000
	Pred Pos.	Pred Neg.
	Real Pos.	0	56
	Real Neg.	0	54


Epoch 2 / 3: 100%|██████████| 164/164 [06:53<00:00,  2.52s/it]


Epoch 2/3, Loss: 6850.0000


Epoch 3 / 3: 100%|██████████| 164/164 [06:57<00:00,  2.55s/it]


Epoch 3/3, Loss: 6850.0000


[0.6388576030731201,
 1.109257027565036e-06,
 2.4951466267479974e-14,
 4.024901725350344e-21,
 8.74619375823105e-29,
 5.984540922372116e-34,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 50.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0